In [2]:
from helper_files.feature_selection_helper import *

🔵 LOAD DATASETS

In [3]:
base_dir_path = r"D:\DATA_Myelomy"
clinical_biomarkers_path = r"D:\Clinical_data\Table_clinical_data.csv"

merged = merged_lesions_csv(base_dir_path)
spearman = get_spearman_csv(merged, clinical_biomarkers_path)
kw = get_kruskal_wallis_csv(merged, clinical_biomarkers_path)

🔵 GROUPS THAT CAN HAVE CORRELATED FEATURES

In [4]:
# RANDOM FOREST CLASSIFICATION
rf_class_all = random_forest_classifier_selected_features(merged, clinical_biomarkers_path)
rf_class_significant = random_forest_classifier_significant_selected_features(merged, kw, clinical_biomarkers_path)

# RANDOM FOREST REGRESSION
rf_reg_all = random_forest_regressor_selected_features(merged, clinical_biomarkers_path)
rf_reg_significant = random_forest_regressor_significant_selected_features(merged, spearman, clinical_biomarkers_path)

# LASSO REGRESSION
lasso_reg_all = lasso_regression_selected_features(merged, clinical_biomarkers_path)
lasso_reg_significant = lasso_regression_significant_selected_features(merged, spearman, clinical_biomarkers_path)

# MUTUAL INFO
mi_class_all = mutual_information_selected_features(merged, clinical_biomarkers_path)
mi_reg_all = mutual_information_selected_features(merged, clinical_biomarkers_path, clinical_column_name='Stage')

# ONE BEST FEATURE FROM EVERY FEATURE GROUP
best_one_group_kw = best_feature_from_each_feature_group(kw)
best_one_group_spearman = best_feature_from_each_feature_group(spearman)

# FEATURES SELECTED BY SPEARMAN_CORR & H_ALL (K-W)
selected_kw = selected_kruskal_wallis_features(kw)
selected_spearman = selected_spearman_features(spearman)

🔵 COUNT TABLES

In [9]:
all_dicts = [selected_kw, selected_spearman, best_one_group_kw, best_one_group_spearman, rf_class_significant, rf_reg_significant,
             lasso_reg_significant, rf_class_all, rf_reg_all, lasso_reg_all, mi_class_all, mi_reg_all]

dataset_names = all_dicts[0].keys()
result = {ds: Counter() for ds in dataset_names}

for d in all_dicts:
    for dataset_name, feature_list in d.items():
        result[dataset_name].update(feature_list)

tables = {}
for dataset_name, counter in result.items():
    df = pd.DataFrame(counter.items(), columns=["Feature", "Count"])
    df = df.sort_values("Count", ascending=False).reset_index(drop=True)

    total = len(all_dicts)
    df["Ratio[%]"] = round((df["Count"] / total) * 100, 1)
    tables[dataset_name] = df

# Best features merged through methods
for name, df in tables.items():
    print(f"----------------------------------------{name}--------------------------------------------")
    print(df[df["Count"] > 1])
    print()

----------------------------------------BMD--------------------------------------------
                                            Feature  Count  Ratio[%]
0                      original_firstorder_Kurtosis      9      75.0
1            original_firstorder_InterquartileRange      6      50.0
2   original_glszm_GrayLevelNonUniformityNormalized      6      50.0
3                      original_firstorder_Skewness      6      50.0
4                           original_ngtdm_Contrast      5      41.7
5                                 original_glcm_Idn      5      41.7
6                                gradient_glcm_Imc2      4      33.3
7                                gradient_glcm_Imc1      4      33.3
8                  gradient_gldm_DependenceVariance      4      33.3
9                         original_glcm_JointEnergy      4      33.3
10           gradient_glrlm_LowGrayLevelRunEmphasis      3      25.0
11                        original_ngtdm_Complexity      3      25.0
12             

🔵 GROUPS WITHOUT CORRELATED FEATURES

In [10]:
# FEATURES FILTERED BY CORRELATION BETWEEN THEM
filtered_kw = filtered_features_kruskal_wallis(merged, kw)
filtered_spearman = filtered_features_spearman(merged, spearman)

# MRMR CLASSIFICATION (2 types)
mrmr_significant_class_01 = mrmr_classification_significant_selected_features(merged, kw, clinical_biomarkers_path)
mrmr_significant_class_02 = mrmr_fe_classification_significant_selected_features(merged, kw, clinical_biomarkers_path)
mrmr_all_class_01 = mrmr_classification_selected_features(merged, clinical_biomarkers_path)
mrmr_all_class_02 = mrmr_fe_classification_selected_features(merged, clinical_biomarkers_path)

# MRMR REGRESSION
mrmr_significant_reg = mrmr_regression_significant_selected_features(merged, spearman, clinical_biomarkers_path)
mrmr_all_reg = mrmr_regression_selected_features(merged, clinical_biomarkers_path)

100%|██████████| 20/20 [00:00<00:00, 26.84it/s]


🔵 COUNT TABLES

In [12]:
all_dicts = [filtered_kw, filtered_spearman, mrmr_significant_class_01, mrmr_significant_class_02, mrmr_significant_reg,
             mrmr_all_class_01, mrmr_all_class_02, mrmr_all_reg]

dataset_names = all_dicts[0].keys()
result = {ds: Counter() for ds in dataset_names}
for d in all_dicts:
    for dataset_name, feature_list in d.items():
        result[dataset_name].update(feature_list)

tables = {}
for dataset_name, counter in result.items():
    df = pd.DataFrame(counter.items(), columns=["Feature", "Count"])
    df = df.sort_values("Count", ascending=False).reset_index(drop=True)

    total = len(all_dicts)
    df["Ratio[%]"] = round((df["Count"] / total) * 100, 1)
    tables[dataset_name] = df

# NOT REDUNDANT features merged through methods
for name, df in tables.items():
    print(f"-------------------------------------{name}-------------------------------------------")
    print(df[df["Count"] > 1])
    print()

-------------------------------------BMD-------------------------------------------
                                              Feature  Count  Ratio[%]
0                        original_firstorder_Kurtosis      8     100.0
1                        original_firstorder_Skewness      8     100.0
2                             original_ngtdm_Contrast      6      75.0
3                    gradient_firstorder_10Percentile      6      75.0
4              original_firstorder_InterquartileRange      5      62.5
5                          gradient_firstorder_Median      5      62.5
6              gradient_glrlm_LowGrayLevelRunEmphasis      4      50.0
7                                  gradient_glcm_Imc1      4      50.0
8     gradient_glszm_GrayLevelNonUniformityNormalized      4      50.0
9                    gradient_gldm_DependenceVariance      4      50.0
10  original_gldm_SmallDependenceHighGrayLevelEmph...      3      37.5
11                             original_glcm_Contrast      3    